# Lab Experiment 1 — Multimodal Data Ingestion for AI in Healthcare

**Course:** CSET343 — AI in Healthcare | B.Tech Year 4, Semester VII

**Objective:** Implementation of a program to read tabular, textual, image, signal, and medical
data in different formats.

**Datasets used (fetched directly from source):**
- **Tabular:** UCI Cleveland Heart Disease dataset (via `ucimlrepo`, id = 45)
- **Signal:** MIT-BIH Arrhythmia Database, record 100 (via `wfdb`, from PhysioNet)
- **Text:** a sample clinical note (typed in directly, as suggested in the lab brief)
- **Image:** a public-domain chest X-ray (via `requests`, from Wikimedia Commons)


In [ ]:
# 1. Environment setup
!pip install -q ucimlrepo wfdb pydicom

import io
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (8, 5)


## 2. Tabular Data Ingestion — Heart Disease (Cleveland) Dataset

**Format:** CSV | **Library:** `pandas`, `ucimlrepo`

`ucimlrepo` is the official UCI Machine Learning Repository package — it downloads the dataset
directly from `archive.ics.uci.edu` and returns it as a pandas DataFrame.


In [ ]:
from ucimlrepo import fetch_ucirepo

heart_disease = fetch_ucirepo(id=45)

X = heart_disease.data.features
y = heart_disease.data.targets

tabular_df = pd.concat([X, y], axis=1)
print(tabular_df.shape)
tabular_df.head()


In [ ]:
# Inspect schema
tabular_df.dtypes


In [ ]:
# Missing values (the 'ca' and 'thal' columns contain a few missing entries)
tabular_df.isnull().sum()


In [ ]:
# Handle missing values — drop the few rows with missing 'ca'/'thal'
tabular_df = tabular_df.dropna()

# Map target to binary: 0 = no heart disease, 1 = heart disease present (any severity 1-4)
tabular_df['target'] = (tabular_df['num'] > 0).astype(int)
tabular_df = tabular_df.drop(columns=['num'])

tabular_df['target'].value_counts()


In [ ]:
# Exploratory data analysis
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
tabular_df['age'].hist(ax=axes[0], bins=15, color='steelblue')
axes[0].set_title('Age distribution')

tabular_df['target'].value_counts().plot(kind='bar', ax=axes[1], color='indianred')
axes[1].set_title('Heart disease presence (0=no, 1=yes)')

tabular_df['chol'].hist(ax=axes[2], bins=15, color='seagreen')
axes[2].set_title('Cholesterol distribution')

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(tabular_df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature correlation matrix')
plt.tight_layout()
plt.show()


In [ ]:
# Baseline classifier: Logistic Regression
X_train, X_test, y_train, y_test = train_test_split(
    tabular_df.drop(columns=['target']), tabular_df['target'],
    test_size=0.2, random_state=42, stratify=tabular_df['target'])

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC :", roc_auc_score(y_test, y_prob))


In [ ]:
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['No disease', 'Disease']).plot(cmap='Blues')
plt.title('Confusion Matrix — Heart Disease Classifier')
plt.show()


## 3. Signal Data Ingestion — ECG (MIT-BIH Arrhythmia Database)

**Format:** WFDB | **Library:** `wfdb`

We fetch record `"100"` directly from PhysioNet's MIT-BIH Arrhythmia Database.


In [ ]:
import wfdb

record = wfdb.rdrecord('100', pn_dir='mitdb', sampto=5000)
annotation = wfdb.rdann('100', 'atr', pn_dir='mitdb', sampto=5000)

signal = record.p_signal[:, 0]
fs = record.fs
r_peaks = annotation.sample

print(f"Sampling rate: {fs} Hz")
print(f"Signal length: {len(signal)} samples ({len(signal)/fs:.1f} s)")
print(f"Number of annotated beats: {len(r_peaks)}")


In [ ]:
t = np.arange(len(signal)) / fs

plt.figure(figsize=(12, 4))
plt.plot(t, signal, color='crimson', linewidth=0.8)
plt.plot(r_peaks / fs, signal[r_peaks], 'ko', markersize=4, label='Annotated beats (R-peaks)')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude (mV)')
plt.title('MIT-BIH Record 100 — ECG segment with beat annotations')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Basic heart-rate / beat-interval statistics
rr_intervals_sec = np.diff(r_peaks) / fs
heart_rate_bpm = 60 / rr_intervals_sec

print(f"Mean R-R interval: {rr_intervals_sec.mean()*1000:.1f} ms")
print(f"SDNN (std of R-R intervals): {rr_intervals_sec.std()*1000:.1f} ms")
print(f"Mean heart rate: {heart_rate_bpm.mean():.1f} bpm")


## 4. Textual Data Demonstration — Clinical Note

**Format:** Plain text | **Libraries:** basic Python string processing / `nltk`

A small sample clinical note (as suggested in the lab brief), tokenized with simple word-frequency
analysis to extract key terms.


In [ ]:
clinical_note = """
Patient is a 65-year-old male presenting with intermittent chest pain radiating to the left arm,
worsened by exertion and relieved by rest. Reports associated shortness of breath and mild
diaphoresis. History of hypertension and type 2 diabetes mellitus. No prior myocardial infarction.
ECG shows non-specific ST-T wave changes. Resting blood pressure 148/92 mmHg. Plan: obtain
troponin levels, initiate aspirin, and schedule stress echocardiography to further evaluate for
coronary artery disease.
"""

stopwords = set("""a an the is are was were be been being of to in on for and or with by at as
from this that these those it its patient he she his her not no""".split())

cleaned = " ".join(clinical_note.split())
tokens = [t.strip(".,;:()").lower() for t in cleaned.split()]
tokens = [t for t in tokens if t and t not in stopwords]

term_freq = pd.Series(tokens).value_counts()
print(f"Token count after stopword removal: {len(tokens)}")
term_freq.head(10)


In [ ]:
term_freq.head(15).plot(kind='barh', color='darkorange')
plt.title('Top clinical-note terms by frequency')
plt.xlabel('Count')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 5. Image Data Demonstration — Chest X-ray

**Format:** JPEG | **Library:** `Pillow`

We fetch a public-domain sample chest X-ray directly from Wikimedia Commons.


In [ ]:
image_url = "https://upload.wikimedia.org/wikipedia/commons/a/a4/Chest_X-Ray.jpg"

response = requests.get(image_url)
xray_img = Image.open(io.BytesIO(response.content))

print("Format:", xray_img.format)
print("Size (W x H):", xray_img.size)
print("Mode:", xray_img.mode)

plt.imshow(xray_img, cmap='gray')
plt.title('Chest X-ray (Wikimedia Commons, public domain)')
plt.axis('off')
plt.show()


In [ ]:
# Resize / normalize
xray_resized = xray_img.convert('L').resize((256, 256))
xray_array = np.array(xray_resized).astype(np.float32)
xray_norm = (xray_array - xray_array.min()) / (xray_array.max() - xray_array.min())

plt.imshow(xray_norm, cmap='gray')
plt.title('Resized & normalized (256x256, grayscale)')
plt.axis('off')
plt.show()

print("Normalized pixel range:", xray_norm.min(), "-", xray_norm.max())


## 6. Integration Discussion

- **Tabular + Signal:** ECG-derived features (mean heart rate, R-R interval SDNN) could be added
  as extra columns to the tabular heart-disease feature table, giving the model direct
  physiological signal information rather than relying only on `thalach` (max heart rate achieved)
  as a single summary value.
- **Tabular + Text:** Symptom keywords extracted from clinical notes (e.g. "chest pain",
  "shortness of breath") could be one-hot encoded and merged into the same patient row, adding
  context that structured fields alone don't capture (onset, radiation pattern, triggers).
- **Fusion strategy:** All modalities would be aligned by a common patient/subject ID (and visit
  timestamp where relevant) before being passed to a downstream classifier — either a single model
  on the fused feature vector, or a multimodal network with a separate encoder per modality.
